<a href="https://colab.research.google.com/github/MInesGomes/AI-Project2026/blob/main/AI_ComputerVision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install tensorflow

## Prepare and Load Data

Since this is an image classification model, we'll need image data. For demonstration, I'll set up `ImageDataGenerator` for data augmentation and loading, assuming you'll point it to your image directories (e.g., `train_dir` and `validation_dir`) which should contain subdirectories for each class (e.g., 'damaged' and 'intact').

In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
from PIL import Image
import numpy as np # Import numpy
import shutil # Import shutil for directory cleanup

# --- Create Dummy Directories and Images for Demonstration ---
# In a real scenario, you would replace these with your actual dataset paths.
base_dir = 'data'
# Ensure a clean slate by removing the directory if it exists
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir)

train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'validation')

# Create subdirectories for 'damaged' and 'intact' classes
for directory in [train_dir, validation_dir]:
    os.makedirs(os.path.join(directory, 'damaged'), exist_ok=True)
    os.makedirs(os.path.join(directory, 'intact'), exist_ok=True)

# Simulate some dummy image files (e.g., create empty files)
# This is just so flow_from_directory doesn't throw an error due to empty directories.
def create_dummy_image(path, filename, size=(150, 150)):
    # Create a simple red image using numpy and PIL
    img_array = np.full((size[0], size[1], 3), [255, 0, 0], dtype=np.uint8) # Red image
    img = Image.fromarray(img_array)
    img.save(os.path.join(path, filename), 'JPEG')

for i in range(5):
    create_dummy_image(os.path.join(train_dir, 'damaged'), f'img_{i}.jpg')
    create_dummy_image(os.path.join(train_dir, 'intact'), f'img_{i}.jpg')
    create_dummy_image(os.path.join(validation_dir, 'damaged'), f'val_img_{i}.jpg')
    create_dummy_image(os.path.join(validation_dir, 'intact'), f'val_img_{i}.jpg')


# --- Image Data Generators ---
# Rescale images by 1./255 for normalization
train_datagen = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)

# Flow training images in batches of 20 using train_datagen generator
train_generator = train_datagen.flow_from_directory(
    train_dir, # This is the target directory
    target_size=(150, 150), # All images will be resized to 150x150
    batch_size=20,
    class_mode='binary' # Since we use binary_crossentropy loss, we need binary labels
)

# Flow validation images in batches of 20 using validation_datagen generator
validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(150, 150),
    batch_size=20,
    class_mode='binary'
)

Found 10 images belonging to 2 classes.
Found 10 images belonging to 2 classes.


In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Define the Architecture
def build_damage_detector():
    model = models.Sequential([
        # Convolutional layers detect patterns (cracks, dents)
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Flatten the 2D patterns into a 1D vector for the final decision
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid') # Output: 0 (Intact) or 1 (Damaged)
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# 2. Simulate Loading Data
# In a real scenario, you'd use flow_from_directory to point to your image folders
model = build_damage_detector()
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 82944)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │     5,308,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,327,937 (20.32 MB)

 Trainable params: 5,327,937 (20.32 MB)

 Non-trainable params: 0 (0.00 B)

## Train the Model

Now, let's train the model using the `fit` method. This process involves feeding the training data to the model, adjusting its weights based on the loss function, and monitoring its performance on the validation set. You'll typically run this for a number of `epochs` (passes over the entire dataset).

In [8]:
history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    epochs=5, # You can increase this for better training
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size
)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5000 - loss: 0.6959 - val_accuracy: 0.5000 - val_loss: 5.3088
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 447ms/step - accuracy: 0.5000 - loss: 5.3088 - val_accuracy: 0.5000 - val_loss: 1.2320
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 856ms/step - accuracy: 0.5000 - loss: 1.2320 - val_accuracy: 0.5000 - val_loss: 0.7615
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 752ms/step - accuracy: 0.5000 - loss: 0.7615 - val_accuracy: 0.5000 - val_loss: 0.7730
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 909ms/step - accuracy: 0.5000 - loss: 0.7730 - val_accuracy: 0.5000 - val_loss: 0.7623


## Evaluate the Model

After training, we evaluate the model to get a final measure of its performance on unseen data. The `evaluate` method will return the loss and any metrics specified during `compile` (in this case, accuracy).

In [9]:
loss, accuracy = model.evaluate(validation_generator)
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.5000 - loss: 0.7623
Validation Loss: 0.7623
Validation Accuracy: 0.5000


## Meaning of the Results

After running the evaluation code above, you will see two key metrics: `Validation Loss` and `Validation Accuracy`.

*   **Validation Loss**: This value represents how well the model performed on the validation set, with respect to the `binary_crossentropy` loss function. A lower loss value generally indicates a better-performing model. It quantifies the 'error' in the model's predictions.

*   **Validation Accuracy**: This is the proportion of correctly classified images in the validation set. An accuracy of 1.0 (or 100%) means the model classified every image correctly, while 0.5 (or 50%) would mean it's performing no better than random guessing for a balanced binary classification task. For a damage detection model, a high validation accuracy means the model is good at distinguishing between 'damaged' and 'intact' vehicles.